# imports

In [ ]:
from IPython.display import display, Markdown

In [ ]:
import uuid
from pathlib import Path

from finrag.chunking import DoclingHybridChunker
from finrag.chunk_postprocess import (
    DocumentContextPostprocessor,
    HeuristicSummaryPostprocessor,
    SectionLinkPostprocessor,
    YahooFinanceCompanyNameResolver,
)

# load markdown and chunk with default args. bad, tables are broken up

In [ ]:
path = Path(
    "/home/mlin/repos/z_scratch/financial-rag/data/sec_filings/processed_markdown/ALAB_000173629724000006_10-Q_2024-05-08.md"
)
doc_id = str(uuid.uuid4())
chunker = DoclingHybridChunker(max_tokens=512, overlap_tokens=64)
docling_chunks = chunker.chunk_document(path, doc_id=doc_id)

In [ ]:
print(len(docling_chunks))

In [ ]:
docling_chunks[8:13]

In [ ]:
for chunk in docling_chunks:
    display(Markdown(chunk.text))
    display(Markdown("---"))

# try increasing chunk size to 1024? 
# another big problem is it replaced the markdown table delimiter chars like | with another symbol.

In [ ]:
path = Path(
    "/home/mlin/repos/z_scratch/financial-rag/data/sec_filings/processed_markdown/ALAB_000173629724000006_10-Q_2024-05-08.md"
)
doc_id = str(uuid.uuid4())
chunker = DoclingHybridChunker(max_tokens=1024, overlap_tokens=128)
docling_chunks = chunker.chunk_document(path, doc_id=doc_id)

In [ ]:
len(docling_chunks)

In [ ]:
for chunk in docling_chunks:
    display(Markdown(chunk.text))
    display(Markdown("---"))

# trying GPT5.2 High's Table Preserving Chunker -> while it does seem to preserves tables, one downside is it leaves too many orphan chunks. 

In [ ]:
from finrag.chunking import MarkdownTablePreservingChunker

In [ ]:
path = Path(
    "/home/mlin/repos/z_scratch/financial-rag/data/sec_filings/processed_markdown/ALAB_000173629724000006_10-Q_2024-05-08.md"
)
meta_path = path.parents[1] / "debug" / path.stem / "metadata.json"
doc_id = str(uuid.uuid4())
chunker = MarkdownTablePreservingChunker(max_tokens=1024, overlap_tokens=64, split_tables=False)
chunks = chunker.chunk_document(path, doc_id=doc_id, metadata_json_path=meta_path)

In [ ]:
len(chunks)

In [ ]:
for chunk in chunks:
    display(chunk.headings)
    display(Markdown(chunk.text))
    display(Markdown("---"))
    display(Markdown("---"))

# try fencing up tables in code blocks to avoid Docling from linearizing them -> yes, it does work. maybe this is simpler

In [ ]:
path = Path(
    "/home/mlin/repos/z_scratch/financial-rag/data/sec_filings/processed_markdown/ALAB_000173629724000006_10-Q_2024-05-08.md"
)
doc_id = str(uuid.uuid4())

chunker = DoclingHybridChunker(
    max_tokens=1024, overlap_tokens=64, preprocess_markdown_tables=True, markdown_table_fence_lang="table"
)
chunks = chunker.chunk_document(path, doc_id)

In [ ]:
print(len(chunks))

In [ ]:
for chunk in chunks:
    display(chunk.headings)
    display(Markdown(chunk.text))
    display(Markdown("---"))
    display(Markdown("---"))

# TODO: try olmOCR on PDF and see if it gives more robust results. actually we can do both, then store both chunks into the DB! (with some dedup if possible). this way, we just let more chunks be retrieved at inference time and give the LLM as much info as possible? 

# DONE: improve context expansion -  IMPT: include filing info like: company, ticker, date, type (10Q vs 10K)

GPT5.2 high ideas:
- Store section_id in Qdrant payload and do “parent/section expansion” at query time (retrieve top-k, then fetch more chunks with same section_id to reduce orphaning without bloating every payload).
- Multi-representation indexing: keep index_text for retrieval, but feed the LLM the raw chunk.text plus summary/section_path (already set up).
- Table-specific indexing: generate a compact “table schema string” (column names + row headers) for embedding/BM25, while keeping the full table raw for grounding.

# DONE v1, need to check if effective: parse section hierarchy , map to parent chunks using this hierarchy 
## seems not good? a lot of chunks have section_size := 1 only, which doesn't seem useful. 
## section_path looks decent enough though

# TODO: might need 2nd pass to merge orphan peers. DoclingHybridChunker does NOT fully take care of it. eg chunks[5] is just 1 sentence... it should be merged? or maybe it doesn't know which chunk to merge with (before or after)

# check latest chunker + postprocess pipeline

In [ ]:
chunker = DoclingHybridChunker(
    max_tokens=1024,
    overlap_tokens=64,
    preprocess_markdown_tables=True,
    markdown_table_fence_lang="table",
    chunk_postprocessors=[
        DocumentContextPostprocessor(company_name_resolver=YahooFinanceCompanyNameResolver()),
        SectionLinkPostprocessor(neighbor_window=2),
        HeuristicSummaryPostprocessor(max_summary_chars=300),
    ],
)

path = Path(
    "/home/mlin/repos/z_scratch/financial-rag/data/sec_filings/processed_markdown/ALAB_000173629724000006_10-Q_2024-05-08.md"
)
doc_id = str(uuid.uuid4())


chunks = chunker.chunk_document(path, doc_id)

In [ ]:
print(len(chunks))

In [ ]:
for chunk in chunks:
    display(chunk.headings)
    # display(Markdown(chunk.text))
    display(Markdown(chunk.metadata["index_text"]))
    display(Markdown("---"))
    display(Markdown("---"))

In [ ]:
chunks[55]

In [ ]:
chunks[50].metadata

In [ ]:
for chunk in chunks:
    if not chunk.metadata.get("summary"):
        continue
    display(chunk.headings)
    # display(Markdown(chunk.text))
    display(Markdown(chunk.metadata["index_text"]))
    display(Markdown("---"))
    display(Markdown("---"))